# Preparación y limpieza de los datos de **generación** de residuos


## 1. Carga de datos

In [1]:
import pandas as pd
import numpy as np

In [2]:
residuos = pd.read_csv('../Data/Original/env_wasgen_linear.csv.gz', compression='gzip')

In [3]:
residuos.head()

,DATAFLOW,LAST UPDATE,freq,unit,hazard,nace_r2,waste,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:ENV_WASGEN(1.0),28/10/26 23:00:00,Annual,Kilograms per capita,Hazardous,"Agriculture, forestry and fishing",Primary waste (TOTAL minus SEC),Albania,2022,0.0,d,NaN
1,ESTAT:ENV_WASGEN(1.0),28/10/26 23:00:00,Annual,Kilograms per capita,Hazardous,"Agriculture, forestry and fishing",Primary waste (TOTAL minus SEC),Austria,2010,9.0,NaN,NaN
2,ESTAT:ENV_WASGEN(1.0),28/10/26 23:00:00,Annual,Kilograms per capita,Hazardous,"Agriculture, forestry and fishing",Primary waste (TOTAL minus SEC),Austria,2012,1.0,NaN,NaN
3,ESTAT:ENV_WASGEN(1.0),28/10/26 23:00:00,Annual,Kilograms per capita,Hazardous,"Agriculture, forestry and fishing",Primary waste (TOTAL minus SEC),Austria,2014,1.0,NaN,NaN
4,ESTAT:ENV_WASGEN(1.0),28/10/26 23:00:00,Annual,Kilograms per capita,Hazardous,"Agriculture, forestry and fishing",Primary waste (TOTAL minus SEC),Austria,2016,NaN,NaN,C


In [4]:
residuos.info()

<class 'pandas.DataFrame'>
RangeIndex: 1841298 entries, 0 to 1841297
Data columns (total 12 columns):
 #   Column       Dtype  
---  ------       -----  
 0   DATAFLOW     str    
 1   LAST UPDATE  str    
 2   freq         str    
 3   unit         str    
 4   hazard       str    
 5   nace_r2      str    
 6   waste        str    
 7   geo          str    
 8   TIME_PERIOD  int64  
 9   OBS_VALUE    float64
 10  OBS_FLAG     str    
 11  CONF_STATUS  str    
dtypes: float64(1), int64(1), str(10)
memory usage: 168.6 MB


## 2. Primera "limpieza"
Quitamos las columnas que no aportan nada y renombramos algunas de las columnas.

In [5]:
wasgen_clean = residuos.copy()

# Eliminar columnas innecesarias
wasgen_clean = wasgen_clean.drop(
    columns=['DATAFLOW', 'LAST UPDATE', 'freq']
)

# Renombrar columnas
wasgen_clean = wasgen_clean.rename(columns={
    'TIME_PERIOD': 'year',
    'OBS_VALUE': 'value',
    'OBS_FLAG': 'obs_flag',
    'CONF_STATUS': 'conf_status'
})

Transformamos la columna "unit" en dos columnas: "kg_per_capita" y "toneladas".

In [6]:
index_cols = [
    'hazard',
    'nace_r2',
    'waste',
    'geo',
    'year',
    'obs_flag',
    'conf_status'
]

wasgen_clean = wasgen_clean.pivot(
    index=index_cols,
    columns='unit',
    values='value'
).reset_index()

# Eliminar el nombre del índice de columnas
wasgen_clean.columns.name = None

In [7]:
# Renombramos las nuevas columnas
wasgen_clean = wasgen_clean.rename(columns={
    'Kilograms per capita': 'kg_per_capita',
    'Tonne': 'toneladas',
})

In [8]:
wasgen_clean

,hazard,nace_r2,waste,geo,year,obs_flag,conf_status,kg_per_capita,toneladas
0,Hazardous,"Agriculture, forestry and fishing","Acid, alkaline or saline wastes",Austria,2004,NaN,NaN,0.0,1.0
1,Hazardous,"Agriculture, forestry and fishing","Acid, alkaline or saline wastes",Austria,2006,NaN,NaN,0.0,0.0
2,Hazardous,"Agriculture, forestry and fishing","Acid, alkaline or saline wastes",Austria,2008,NaN,NaN,0.0,0.0
3,Hazardous,"Agriculture, forestry and fishing","Acid, alkaline or saline wastes",Austria,2010,NaN,NaN,0.0,0.0
4,Hazardous,"Agriculture, forestry and fishing","Acid, alkaline or saline wastes",Austria,2012,NaN,NaN,0.0,0.0
...,...,...,...,...,...,...,...,...,...
924442,Non-hazardous,Wholesale of waste and scrap,Wood wastes,United Kingdom,2010,NaN,NaN,1.0,67793.0
924443,Non-hazardous,Wholesale of waste and scrap,Wood wastes,United Kingdom,2012,NaN,NaN,1.0,53494.0
924444,Non-hazardous,Wholesale of waste and scrap,Wood wastes,United Kingdom,2014,NaN,NaN,1.0,52246.0
924445,Non-hazardous,Wholesale of waste and scrap,Wood wastes,United Kingdom,2016,NaN,NaN,0.0,23961.0


# 3. Filtrado de datos
Filtramos los datos a analizar, manteniendo sólo los países de la UE.

In [9]:
paises_ue = [
    "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", 
    "Czechia", "Denmark", "Estonia", "Finland", 
    "France", "Germany", "Greece", "Hungary", "Ireland", 
    "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta", 
    "Netherlands", "Poland", "Portugal", "Romania", "Slovakia", 
    "Slovenia", "Spain", "Sweden", 'European Union - 27 countries (from 2020)'
]

wasgen_clean = wasgen_clean[wasgen_clean['geo'].isin(paises_ue)]

In [10]:
wasgen_clean.head()

,hazard,nace_r2,waste,geo,year,obs_flag,conf_status,kg_per_capita,toneladas
0,Hazardous,"Agriculture, forestry and fishing","Acid, alkaline or saline wastes",Austria,2004,NaN,NaN,0.0,1.0
1,Hazardous,"Agriculture, forestry and fishing","Acid, alkaline or saline wastes",Austria,2006,NaN,NaN,0.0,0.0
2,Hazardous,"Agriculture, forestry and fishing","Acid, alkaline or saline wastes",Austria,2008,NaN,NaN,0.0,0.0
3,Hazardous,"Agriculture, forestry and fishing","Acid, alkaline or saline wastes",Austria,2010,NaN,NaN,0.0,0.0
4,Hazardous,"Agriculture, forestry and fishing","Acid, alkaline or saline wastes",Austria,2012,NaN,NaN,0.0,0.0


Además, quitamos los tipos de residuos que se han quedado obsoletos *(up to 2008)*.

In [11]:
wasgen_clean['waste'].unique()

<StringArray>
[                                                                                                       'Acid, alkaline or saline wastes',
                                                                                                            'Animal and mixed food waste',
                                                                                                        'Animal faeces, urine and manure',
                                                                                                      'Batteries and accumulators wastes',
                                                                                                 'Chemical and medical wastes (subtotal)',
                                                                                                                        'Chemical wastes',
                                                                                                                      'Combustion wastes',
             

In [12]:
wasgen_clean = wasgen_clean[
    ~wasgen_clean['waste'].str.contains('valid up to 2008', case=False, na=False)
]

## 4. Traducción de etiquetas
Se conservan las columnas originales (`geo`, `waste`, etc.) y se crean columnas en castellano.

In [13]:
PAISES_ES = {'Austria': 'Austria', 'Belgium': 'Bélgica', 'Bulgaria': 'Bulgaria', 'Croatia': 'Croacia', 'Cyprus': 'Chipre', 'Czechia': 'Chequia', 'Denmark': 'Dinamarca', 'Estonia': 'Estonia', 'Finland': 'Finlandia', 'France': 'Francia', 'Germany': 'Alemania', 'Greece': 'Grecia', 'Hungary': 'Hungría', 'Ireland': 'Irlanda', 'Italy': 'Italia', 'Latvia': 'Letonia', 'Lithuania': 'Lituania', 'Luxembourg': 'Luxemburgo', 'Malta': 'Malta', 'Netherlands': 'Países Bajos', 'Poland': 'Polonia', 'Portugal': 'Portugal', 'Romania': 'Rumanía', 'Slovakia': 'Eslovaquia', 'Slovenia': 'Eslovenia', 'Spain': 'España', 'Sweden': 'Suecia', 'European Union - 27 countries (from 2020)': 'Unión Europea (UE-27)'}

In [14]:
PELIGROSIDAD_ES = {'Hazardous': 'Peligrosos', 'Non-hazardous': 'No peligrosos', 'Hazardous and non-hazardous - total': 'Total (peligrosos y no peligrosos)'}

In [15]:
RESIDUOS_ES = {
    'Acid, alkaline or saline wastes': 'Residuos ácidos, alcalinos o salinos',
    'Animal and mixed food waste': 'Residuos animales y mezclados de alimentos',
    'Animal faeces, urine and manure': 'Heces de animales, orina y estiércol',
    'Batteries and accumulators wastes': 'Residuos de pilas y acumuladores',
    'Chemical and medical wastes (subtotal)': 'Residuos químicos y médicos (subtotal)',
    'Chemical wastes': 'Residuos químicos',
    'Combustion wastes': 'Residuos de combustión',
    'Common sludges': 'Lodos comunes',
    'Discarded equipment (except discarded vehicles and batteries and accumulators waste) (W08 except W081, W0841)': 'Equipos desechados (excepto vehículos desechados y residuos de pilas y acumuladores) (W08 excepto W081, W0841)',
    'Discarded vehicles': 'Vehículos desechados',
    'Dredging spoils': 'Lodos de dragado',
    'Equipment (subtotal, W077+W08A+W081+W0841)': 'Equipos (subtotal, W077+W08A+W081+W0841)',
    'Glass wastes': 'Residuos de vidrio',
    'Health care and biological wastes': 'Residuos sanitarios y biológicos',
    'Household and similar wastes': 'Residuos domésticos y similares',
    'Industrial effluent sludges': 'Lodos de efluentes industriales',
    'Metal wastes, ferrous': 'Residuos metálicos férreos',
    'Metal wastes, mixed ferrous and non-ferrous': 'Residuos metálicos mezclados férreos y no férreos',
    'Metal wastes, non-ferrous': 'Residuos metálicos no férreos',
    'Metallic wastes (W061+W062+W063)': 'Residuos metálicos (W061+W062+W063)',
    'Mineral and solidified wastes (subtotal)': 'Residuos minerales y solidificados (subtotal)',
    'Mineral waste from construction and demolition': 'Residuos minerales de construcción y demolición',
    'Mineral wastes from waste treatment and stabilised wastes': 'Residuos minerales del tratamiento de residuos y residuos estabilizados',
    'Mixed and undifferentiated materials': 'Materiales mezclados e indiferenciados',
    'Mixed ordinary wastes (subtotal, W101+W102+W103)': 'Residuos ordinarios mezclados (subtotal, W101+W102+W103)',
    'Other mineral wastes (W122+W123+W125)': 'Otros residuos minerales (W122+W123+W125)',
    'Paper and cardboard wastes': 'Residuos de papel y cartón',
    'Plastic wastes': 'Residuos de plástico',
    'Primary waste (TOTAL minus SEC)': 'Residuos primarios (TOTAL menos SEC)',
    'Recyclable wastes (subtotal, W06+W07 except W077)': 'Residuos reciclables (subtotal, W06+W07 excepto W077)',
    'Rubber wastes': 'Residuos de caucho',
    'Secondary waste (W033+W103+W128_13)': 'Residuos secundarios (W033+W103+W128_13)',
    'Sludges and liquid wastes from waste treatment': 'Lodos y residuos líquidos del tratamiento de residuos',
    'Soils': 'Tierras y suelos',
    'Sorting residues': 'Residuos de clasificación',
    'Spent solvents': 'Disolventes usados',
    'Textile wastes': 'Residuos textiles',
    'Total waste': 'Total de residuos',
    'Used oils': 'Aceites usados',
    'Vegetal wastes': 'Residuos vegetales',
    'Waste containing PCB': 'Residuos que contienen PCB',
    'Waste excluding major mineral wastes': 'Residuos excluidos los principales residuos minerales',
    'Wood wastes': 'Residuos de madera',
    'Animal and mixed food waste; vegetal wastes (W091+W092)': 'Residuos animales y mezclados de alimentos; residuos vegetales (W091+W092)',
    'Animal and vegetal wastes (subtotal, W091+W092+W093)': 'Residuos animales y vegetales (subtotal, W091+W092+W093)'
}

In [16]:
ACTIVIDADES_ES = {'Agriculture, forestry and fishing': 'Agricultura, ganadería, silvicultura y pesca',
    'All NACE activities plus households': 'Todas las actividades NACE más los hogares',
    'Construction': 'Construcción',
    'Electricity, gas, steam and air conditioning supply': 'Suministro de energía eléctrica, gas, vapor y aire acondicionado',
    'Households': 'Hogares',
    'Manufacture of basic metals and fabricated metal products, except machinery and equipment': 'Metalurgia y fabricación de productos metálicos, excepto maquinaria y equipo',
    'Manufacture of chemical, pharmaceutical, rubber and plastic products': 'Industria química, farmacéutica, y fabricación de productos de caucho y plástico',
    'Manufacture of coke and refined petroleum products': 'Coquerías y refino de petróleo',
    'Manufacture of computer, electronic and optical products, electrical equipment, motor vehicles and other transport equipment': 'Fabricación de productos informáticos, electrónicos y ópticos, material y equipo eléctrico, vehículos de motor y otro material de transporte',
    'Manufacture of food products; beverages and tobacco products': 'Industria de la alimentación, fabricación de bebidas e industria del tabaco',
    'Manufacture of furniture; jewellery, musical instruments, toys; repair and installation of machinery and equipment': 'Fabricación de muebles; joyería, instrumentos musicales, juguetes; reparación e instalación de maquinaria y equipo',
    'Manufacture of other non-metallic mineral products': 'Fabricación de otros productos minerales no metálicos',
    'Manufacture of paper and paper products; printing and reproduction of recorded media': 'Industria del papel; artes gráficas y reproducción de soportes grabados',
    'Manufacture of textiles, wearing apparel, leather and related products': 'Industria textil, confección de prendas de vestir e industria del cuero y del calzado',
    'Manufacture of wood and of products of wood and cork, except furniture; manufacture of articles of straw and plaiting materials': 'Industria de la madera y del corcho, excepto muebles; cestería y espartería',
    'Manufacturing': 'Industria manufacturera',
    'Mining and quarrying': 'Industrias extractivas',
    'Services (except wholesale of waste and scrap)': 'Servicios (excepto comercio al por mayor de chatarra y productos de desecho)',
    'Waste collection, treatment and disposal activities; materials recovery': 'Recogida, tratamiento y eliminación de residuos; valorización',
    'Water collection, treatment and supply; sewerage; remediation activities and other waste management services': 'Captación, depuración y distribución de agua; recogida y tratamiento de aguas residuales; actividades de descontaminación y otros servicios de gestión de residuos',
    'Water supply; sewerage, waste management and remediation activities': 'Suministro de agua, actividades de saneamiento, gestión de residuos y descontaminación',
    'Wholesale of waste and scrap': 'Comercio al por mayor de chatarra y productos de desecho'}

In [17]:
wasgen_clean = wasgen_clean.copy()

wasgen_clean['pais'] = wasgen_clean['geo'].replace(PAISES_ES)
wasgen_clean['peligrosidad'] = wasgen_clean['hazard'].replace(PELIGROSIDAD_ES)
wasgen_clean['residuo'] = wasgen_clean['waste'].replace(RESIDUOS_ES)
wasgen_clean['actividad'] = wasgen_clean['nace_r2'].replace(ACTIVIDADES_ES)

In [18]:
wasgen_clean.head()

,hazard,nace_r2,waste,geo,year,obs_flag,conf_status,kg_per_capita,toneladas,pais,peligrosidad,residuo,actividad
0,Hazardous,"Agriculture, forestry and fishing","Acid, alkaline or saline wastes",Austria,2004,NaN,NaN,0.0,1.0,Austria,Peligrosos,"Residuos ácidos, alcalinos o salinos","Agricultura, ganadería, silvicultura y pesca"
1,Hazardous,"Agriculture, forestry and fishing","Acid, alkaline or saline wastes",Austria,2006,NaN,NaN,0.0,0.0,Austria,Peligrosos,"Residuos ácidos, alcalinos o salinos","Agricultura, ganadería, silvicultura y pesca"
2,Hazardous,"Agriculture, forestry and fishing","Acid, alkaline or saline wastes",Austria,2008,NaN,NaN,0.0,0.0,Austria,Peligrosos,"Residuos ácidos, alcalinos o salinos","Agricultura, ganadería, silvicultura y pesca"
3,Hazardous,"Agriculture, forestry and fishing","Acid, alkaline or saline wastes",Austria,2010,NaN,NaN,0.0,0.0,Austria,Peligrosos,"Residuos ácidos, alcalinos o salinos","Agricultura, ganadería, silvicultura y pesca"
4,Hazardous,"Agriculture, forestry and fishing","Acid, alkaline or saline wastes",Austria,2012,NaN,NaN,0.0,0.0,Austria,Peligrosos,"Residuos ácidos, alcalinos o salinos","Agricultura, ganadería, silvicultura y pesca"


# 5. Limpieza final y exportación
Para exportar los datos, nos quedamos con los datos en castellano.

In [19]:
columnas_a_mantener = [
    'pais', 
    'year', 
    'peligrosidad', 
    'actividad', 
    'residuo', 
    'kg_per_capita', 
    'toneladas', 
    'obs_flag', 
    'conf_status'
]

gen_residuos = wasgen_clean[columnas_a_mantener]

In [20]:
gen_residuos.head()

,pais,year,peligrosidad,actividad,residuo,kg_per_capita,toneladas,obs_flag,conf_status
0,Austria,2004,Peligrosos,"Agricultura, ganadería, silvicultura y pesca","Residuos ácidos, alcalinos o salinos",0.0,1.0,NaN,NaN
1,Austria,2006,Peligrosos,"Agricultura, ganadería, silvicultura y pesca","Residuos ácidos, alcalinos o salinos",0.0,0.0,NaN,NaN
2,Austria,2008,Peligrosos,"Agricultura, ganadería, silvicultura y pesca","Residuos ácidos, alcalinos o salinos",0.0,0.0,NaN,NaN
3,Austria,2010,Peligrosos,"Agricultura, ganadería, silvicultura y pesca","Residuos ácidos, alcalinos o salinos",0.0,0.0,NaN,NaN
4,Austria,2012,Peligrosos,"Agricultura, ganadería, silvicultura y pesca","Residuos ácidos, alcalinos o salinos",0.0,0.0,NaN,NaN


In [21]:
gen_residuos.to_csv("../Data/gen_residuos.csv", index=False)